# Import libreries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

In [2]:
# 1. LOAD & INSPECT

In [3]:
print("=== Step 1: Load & Inspect ===")

# Load the uploaded raw dataset
df = pd.read_csv('raw_loan_dataset.csv')

print("\n--- First 5 Rows (Head) ---")
print(df.head())

print("\n--- DataFrame Summary Info ---")
print(df.info())

print("\n--- Initial Missing Values Count ---")
print(df.isnull().sum())

print("\n🚨 Identified Key Issues in 'raw_loan_dataset.csv':")
print("- 'Income' and 'LoanAmount' contain string symbols like '$' and ',' causing them to be read as object types.")
print("- String categorical columns contain inconsistent dirty labels (e.g., 'Y', 'N', 'yse', 'Approved', 'Rejected', 'NO').")
print("- Multiple columns contain missing values (NaN) that require deterministic imputation.")

=== Step 1: Load & Inspect ===

--- First 5 Rows (Head) ---
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  

--- DataFrame Summary Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   Employment

In [4]:
# 2. CLEAN CURRENCY FORMATTING

In [5]:
print("\n=== Step 2: Clean Currency Formatting ===")

for col in ['Income', 'LoanAmount']:
    # Strip whitespace, strip dollar signs, strip commas, and cast to numeric
    df[col] = df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nData types after currency formatting cleanup:")
print(df[['Income', 'LoanAmount']].dtypes)
print("\nSample check of numeric conversions:")
print(df[['Income', 'LoanAmount']].head())


=== Step 2: Clean Currency Formatting ===

Data types after currency formatting cleanup:
Income        float64
LoanAmount    float64
dtype: object

Sample check of numeric conversions:
     Income  LoanAmount
0  108810.0     25800.0
1   96482.0     11200.0
2   28478.0     12100.0
3   25851.0      7000.0
4   38396.0     10700.0


In [6]:
# 3. FIX CATEGORY ERRORS BEFORE IMPUTATION

In [7]:
print("\n=== Step 3: Fix Category Errors before Imputation ===")

# Normalize strings by converting to lower-case and stripping white-spaces to simplify mapping
for col in ['HasCollateral', 'PreviousDefaults', 'Approved']:
    df[col] = df[col].astype(str).str.lower().str.strip()

# Establish standardized dictionary mappings based on the raw dataset's specific unique values
collateral_map = {'yes': 'Yes', 'y': 'Yes', 'yse': 'Yes', 'no': 'No', 'n': 'No', 'nan': np.nan}
df['HasCollateral'] = df['HasCollateral'].map(collateral_map)
print("\nHasCollateral Value Counts:")
print(df['HasCollateral'].value_counts(dropna=False))

default_map = {'yes': 'Yes', 'y': 'Yes', 'no': 'No', 'n': 'No', 'nan': np.nan}
df['PreviousDefaults'] = df['PreviousDefaults'].map(default_map)
print("\nPreviousDefaults Value Counts:")
print(df['PreviousDefaults'].value_counts(dropna=False))

approved_map = {'yes': 'Yes', 'approved': 'Yes', 'no': 'No', 'rejected': 'No', 'nan': np.nan}
df['Approved'] = df['Approved'].map(approved_map)
print("\nApproved Value Counts:")
print(df['Approved'].value_counts(dropna=False))


=== Step 3: Fix Category Errors before Imputation ===

HasCollateral Value Counts:
HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64

PreviousDefaults Value Counts:
PreviousDefaults
No     85
Yes    14
NaN     4
Name: count, dtype: int64

Approved Value Counts:
Approved
Yes    68
No     35
Name: count, dtype: int64


In [8]:
# 4. IMPUTE MISSING VALUES

In [9]:
print("\n=== Step 4: Impute Missing Values ===")

numeric_cols = ['Income', 'LoanAmount', 'CreditScore', 'EmploymentYears']
categorical_cols = ['HasCollateral', 'PreviousDefaults', 'Approved']

# Numerical feature imputation via column-level median
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical feature imputation via column-level mode
for col in categorical_cols:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)

print("\nMissing values remaining post-imputation (Target = 0):")
print(df.isnull().sum())


=== Step 4: Impute Missing Values ===

Missing values remaining post-imputation (Target = 0):
Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


In [10]:
# 5. REMOVE DUPLICATES

In [11]:
print("\n=== Step 5: Remove Duplicates ===")

rows_before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
rows_after = df.shape[0]

print(f"Row count BEFORE duplicate elimination: {rows_before}")
print(f"Row count AFTER duplicate elimination: {rows_after}")


=== Step 5: Remove Duplicates ===
Row count BEFORE duplicate elimination: 103
Row count AFTER duplicate elimination: 100


In [12]:
# 6. OUTLIERS (IQR CAPPING)

In [13]:

print("\n=== Step 6: Outliers (IQR Capping) ===")

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Non-destructive capping via .clip() keeps rows intact while handling extremes
    df[col] = df[col].clip(lower_bound, upper_bound)

print("\nNumerical data frame description post-IQR capping:")
print(df[numeric_cols].describe())


=== Step 6: Outliers (IQR Capping) ===

Numerical data frame description post-IQR capping:
             Income    LoanAmount  CreditScore  EmploymentYears
count     100.00000    100.000000   100.000000       100.000000
mean    72220.22625  25507.125000   651.100000        12.654500
std     29179.39312  14793.836616    96.218239         7.011164
min     25851.00000   5000.000000   484.000000         0.500000
25%     47964.75000  14400.000000   576.000000         6.275000
50%     69460.50000  20700.000000   638.000000        12.550000
75%     95826.50000  35125.000000   730.500000        17.975000
max    167619.12500  66212.500000   920.000000        25.000000


In [14]:
# 7. LABEL ENCODING

In [15]:
print("\n=== Step 7: Label Encoding ===")

binary_label_map = {'Yes': 1, 'No': 0}
for col in ['Approved', 'HasCollateral', 'PreviousDefaults']:
    df[col] = df[col].map(binary_label_map)

print("\nClass distribution of encoded Target feature ('Approved'):")
print(df['Approved'].value_counts())


=== Step 7: Label Encoding ===

Class distribution of encoded Target feature ('Approved'):
Approved
1    66
0    34
Name: count, dtype: int64


In [17]:
# 8. CLASS BALANCE CHECK

In [18]:
print("\n=== Step 8: Class Balance Check ===")

counts = df['Approved'].value_counts()
proportions = df['Approved'].value_counts(normalize=True)

print("Class Frequencies:\n", counts)
print("Class Ratios:\n", proportions)

# Operational check for evaluation baseline accuracy metrics
if proportions.min() < 0.35:
    print("\n⚠️ Note: The target classes display notable imbalance. Relying on Accuracy alone could be misleading; optimize for F1-Score instead.")
else:
    print("\n✅ Target class distribution is balanced enough for Accuracy to serve as an evaluation baseline.")


=== Step 8: Class Balance Check ===
Class Frequencies:
 Approved
1    66
0    34
Name: count, dtype: int64
Class Ratios:
 Approved
1    0.66
0    0.34
Name: proportion, dtype: float64

⚠️ Note: The target classes display notable imbalance. Relying on Accuracy alone could be misleading; optimize for F1-Score instead.


In [19]:
# 9. FEATURE ENGINEERING (NO LEAKAGE)

In [20]:
print("\n=== Step 9: Feature Engineering ===")

# Created entirely from inputs to ensure zero mathematical relationship to 'Approved'
df['DebtToIncome'] = df['LoanAmount'] / (df['Income'] + 1e-5)
df['IncomePerYearEmployed'] = df['Income'] / (df['EmploymentYears'] + 1)

print("\nEngineered columns sample snapshot:")
print(df[['DebtToIncome', 'IncomePerYearEmployed']].head())


=== Step 9: Feature Engineering ===

Engineered columns sample snapshot:
   DebtToIncome  IncomePerYearEmployed
0      0.237111           51814.285714
1      0.116084            6030.125000
2      0.424889            4449.687500
3      0.270783            1389.838710
4      0.278675            3555.185185


In [22]:
# 10. FEATURE SCALING (RESEARCH & CHOOSE)

In [23]:
print("\n=== Step 10: Feature Scaling ===")

# Include original continuous metrics along with engineered metrics
continuous_features = numeric_cols + ['DebtToIncome', 'IncomePerYearEmployed']

# Instantiate RobustScaler instead of StandardScaler
scaler = RobustScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

print("\nFirst 5 entries of scaled features:")
print(df[continuous_features].head())


=== Step 10: Feature Scaling ===

First 5 entries of scaled features:
     Income  LoanAmount  CreditScore  EmploymentYears  DebtToIncome  \
0  0.822149    0.246080    -0.653722        -0.978632     -0.445244   
1  0.564574   -0.458384    -0.737864         0.209402     -0.912749   
2 -0.856268   -0.414958     0.000000        -0.611111      0.280113   
3 -0.911156   -0.661037    -0.498382         0.431624     -0.315175   
4 -0.649046   -0.482509    -0.718447        -0.235043     -0.284688   

   IncomePerYearEmployed  
0               8.274536  
1               0.153994  
2              -0.126321  
3              -0.669033  
4              -0.284975  


In [24]:
print("\n=== Step 11: Final Checks & Save ===")

print("\n--- Final Integrity Summary ---")
print(df.info())

print("\n--- Post-Pipeline Null Check (Expected All 0) ---")
print(df.isnull().sum())

print("\n--- Sample of Final Output Cleaned Elements ---")
print(df.head())

# Save dataset out to required file name for part B2 ingestion
df.to_csv('clean_loan_dataset.csv', index=False)
print("\nDataset compiled and successfully written out to 'clean_loan_dataset.csv'. Pipeline complete.")


=== Step 11: Final Checks & Save ===

--- Final Integrity Summary ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.2 KB
None

--- Post-Pipeline Null Check (Expected All 0) ---
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefau